# 🕵️ Fraud Detection Model – Evaluation & Analysis
**Run each cell in order (Runtime → Run all).**

You need two files:
1. `cleaned_fraud_data...csv` – features + `is_fraud`
2. `prediction_results...csv` – model output (same row order)

**Sections:** 1 Upload · 2 Load & merge · 3 Metrics · 4 Threshold tuning · 5 Error analysis · 6 Plots · 7 Export

## 1. Setup & upload files

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

try:                                    # running in Colab
    from google.colab import files
    print("Select BOTH csv files in the dialog 👇")
    files.upload()
    IN_COLAB = True
except ImportError:                     # running locally
    IN_COLAB = False

Select BOTH csv files in the dialog 👇


## 2. Load & merge
The two files line up row-for-row, so we join by position and verify the labels match.

In [ ]:
DATA = [f for f in glob.glob("*.csv") if "fraud_data" in f][0]
PRED = [f for f in glob.glob("*.csv") if "prediction" in f][0]
print("Data file:", DATA, "\nPred file:", PRED)

data = pd.read_csv(DATA)
pred = pd.read_csv(PRED)
assert len(data) == len(pred), "Row counts differ!"
assert (data["is_fraud"].values == pred["actual_fraud"].values).all(), "Labels misaligned!"

df = pd.concat([data, pred.drop(columns=["actual_fraud"])], axis=1)

# Rebuild one 'category' column from the one-hot columns (all False = dropped baseline category)
cat_cols = [c for c in df.columns if c.startswith("category_")]
df["category"] = (df[cat_cols].idxmax(axis=1).str.replace("category_", "")
                  .where(df[cat_cols].any(axis=1), "baseline"))

y, yhat, prob = df["is_fraud"], df["predicted_fraud"], df["fraud_probability"]
print(f"\nRows: {len(df):,} | Fraud cases: {y.sum()} ({y.mean()*100:.3f}%)")
df.head()

## 3. Overall metrics
⚠️ Fraud is only ~0.17% of rows, so **accuracy is misleading** – focus on precision, recall, F1 and PR-AUC.

In [ ]:
cm = confusion_matrix(y, yhat)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn:,}  FP={fp}  FN={fn}  TP={tp}\n")
print(classification_report(y, yhat, digits=4, target_names=["Legit", "Fraud"]))
print("ROC-AUC:", round(roc_auc_score(y, prob), 4))
print("PR-AUC :", round(average_precision_score(y, prob), 4))
print(f"\nBaseline check – always predicting 'legit' gives {(1-y.mean())*100:.2f}% accuracy")

## 4. Threshold tuning
The default 0.5 cutoff misses ~27% of fraud. Let's find a better one.

In [ ]:
prec, rec, thr = precision_recall_curve(y, prob)
f1 = 2 * prec * rec / (prec + rec + 1e-12)
best = f1[:-1].argmax()
print(f"Best-F1 threshold: {thr[best]:.4f}  "
      f"(precision={prec[best]:.3f}, recall={rec[best]:.3f}, F1={f1[best]:.3f})")

# Compare default vs tuned threshold
for name, t in [("Default 0.50", 0.5), ("Best-F1", thr[best])]:
    p_ = (prob >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y, p_).ravel()
    print(f"{name:13s} -> caught {tp_}/{tp_+fn_} frauds | false alarms {fp_}")

In [ ]:
# ---- Cost-based threshold: CHANGE THESE to your real business costs ----
COST_FN = 500   # cost of one MISSED fraud
COST_FP = 10    # cost of one FALSE ALARM
# -------------------------------------------------------------------------

rows = []
for t in np.linspace(0.01, 0.99, 99):
    p_ = (prob >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y, p_).ravel()
    rows.append((t, fp_, fn_, tp_, fp_ * COST_FP + fn_ * COST_FN))
cost = pd.DataFrame(rows, columns=["threshold", "FP", "FN", "TP", "total_cost"])

print("Cheapest threshold:")
display(cost.loc[[cost.total_cost.idxmin()]])

plt.figure(figsize=(8, 4))
plt.plot(cost.threshold, cost.total_cost)
plt.axvline(cost.loc[cost.total_cost.idxmin(), "threshold"], ls="--", c="red")
plt.title("Total cost vs threshold"); plt.xlabel("Threshold"); plt.ylabel("Cost")
plt.show()

## 5. Error analysis – where does the model fail?

In [ ]:
df["error_type"] = np.select(
    [(y == 1) & (yhat == 1), (y == 0) & (yhat == 1), (y == 1) & (yhat == 0)],
    ["TP", "FP", "FN"], default="TN")

print("Mean feature values by outcome (FN = missed fraud, FP = false alarm):")
df.groupby("error_type")[["log_amt", "age", "distance_km", "trans_hour", "city_pop"]].mean().round(3)

In [ ]:
by_cat = df.groupby("category").apply(
    lambda g: pd.Series({"n": len(g), "fraud": g.is_fraud.sum(),
                         "caught": ((g.is_fraud == 1) & (g.predicted_fraud == 1)).sum()}),
    include_groups=False)
by_cat["fraud_rate_%"] = (by_cat.fraud / by_cat.n * 100).round(3)
by_cat["recall"] = (by_cat.caught / by_cat.fraud.replace(0, np.nan)).round(3)
print("Fraud rate & recall by category (small counts – interpret with care):")
by_cat.sort_values("fraud", ascending=False)

In [ ]:
hourly = df.groupby("trans_hour").is_fraud.agg(["mean", "sum"])
hourly["mean"] *= 100
hourly.columns = ["fraud_rate_%", "fraud_count"]
hourly.round(3).T

## 6. Plots

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 10))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[0, 0],
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
ax[0, 0].set(title="Confusion Matrix", xlabel="Predicted", ylabel="Actual")

fpr, tpr, _ = roc_curve(y, prob)
ax[0, 1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y, prob):.3f}")
ax[0, 1].plot([0, 1], [0, 1], "--", c="grey")
ax[0, 1].set(title="ROC Curve", xlabel="False Positive Rate", ylabel="True Positive Rate")
ax[0, 1].legend()

ax[0, 2].plot(rec, prec)
ax[0, 2].set(title=f"Precision-Recall (AP={average_precision_score(y, prob):.3f})",
             xlabel="Recall", ylabel="Precision")

sns.histplot(data=df, x="fraud_probability", hue="is_fraud", bins=50, ax=ax[1, 0])
ax[1, 0].set_yscale("log"); ax[1, 0].set_title("Score distribution (log y)")

sns.boxplot(data=df, x="is_fraud", y="log_amt", ax=ax[1, 1])
ax[1, 1].set_title("log(amount) by class")

(df.groupby("trans_hour").is_fraud.mean() * 100).plot(kind="bar", ax=ax[1, 2])
ax[1, 2].set(title="Fraud rate % by hour", xlabel="Hour", ylabel="%")

plt.tight_layout()
plt.savefig("fraud_analysis.png", dpi=150)
plt.show()

## 7. Export results
Downloads the chart plus the missed frauds and false alarms for manual review.

In [ ]:
df[df.error_type == "FN"].to_csv("missed_fraud_FN.csv", index=False)
df[df.error_type == "FP"].to_csv("false_alarms_FP.csv", index=False)
print("Missed frauds:", (df.error_type == "FN").sum(), "| False alarms:", (df.error_type == "FP").sum())

if IN_COLAB:
    for f in ["fraud_analysis.png", "missed_fraud_FN.csv", "false_alarms_FP.csv"]:
        files.download(f)

## ✅ Key takeaways (from the data you provided)
- Model is **precise (0.94) but under-detects fraud (recall 0.73)** – 41 of 150 frauds missed.
- Lowering the threshold to ~0.35 lifts recall to ~0.85 with precision ~0.91.
- Fraud clusters **10pm–3am**; missed frauds skew to lower amounts and later evening hours.
- Some categories have only 3–5 fraud cases, so their recall numbers are noisy.
- Check that this data was **not used for training**, otherwise metrics are optimistic.